# Feature — Content-Profile Similarity (enhanced Feature 3)

**Purpose.** Personalise through the *text* of the products a user has touched, not only their IDs. Steps 2–4b personalise through exact item / brand / category IDs (`uc_*`, `ubrand_*`, `ucat*_*`) and through a 64-d LSA profile (`u_sem_lsa_7d`, `u_sem_lsa_all`). Exact IDs miss two phone cases from different shops and leaf categories that share most of their title terms, and the LSA profile keeps only a small part of the TF-IDF variance (Step 4d lists the scalar semantic profiles as drop candidates). This notebook scores every candidate against the user's history in the sparse TF-IDF space itself.

**What it adds** (columns in §3), compared with one cosine to one averaged profile:
1. richer item text: name terms plus field-prefixed brand-name and level-3/4 category-name terms;
2. nearest-neighbour similarity (`cps_max`), so a user with several interests is not blurred into one average;
3. a query-conditioned profile (`cps_query`) that re-weights the history by its relevance to the current query;
4. funnel-stage profiles (cart + order, order only) and the age of the best-matching history product, aimed at the purchase cut-point;
5. long (90-day) and short (1-day) horizons as separate columns;
6. `cps_top3_new`, similarity to the nearest history products *other than the candidate itself*, which repeat encounters cannot inflate.

**Contract.** Run after Steps 1–4b. This notebook does **not** modify `features.parquet` or any earlier output. It writes `processed/feature_content_profile/`: the feature table keyed by `user, wid` (in `candidates.parquet` row order) and, per run, the metrics and a manifest. Model comparisons use the train and validation splits only; earlier steps already used the test split for feature analysis, so it is not used here to select features.

Set `JD_SMOKE=1` for a quick local run on ~2% of users. The full run needs a High-RAM Colab runtime (TF-IDF over all slate and history products); no GPU is needed.

In [ ]:
# In Colab, run this once if dependencies are missing:
# %pip install lightgbm pyarrow pandas numpy scipy
import os, json, time, gc, platform, importlib.metadata
from pathlib import Path
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pyarrow as pa, pyarrow.compute as pc, pyarrow.csv as pacsv

try:                                    # Colab: data lives on the shared Drive
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/JDSearch/Raw Data')
except ImportError:                     # local run: set JD_DATA_DIR or use the default download location
    DATA_DIR = Path(os.environ.get('JD_DATA_DIR', '~/Downloads/JDsearch')).expanduser()
PROCESSED = DATA_DIR / 'processed'

SMOKE = os.environ.get('JD_SMOKE') == '1'      # local smoke test: ~2% of users
SEED = 42
SEEDS = [42, 43, 44]                           # LightGBM seeds for the matched comparisons
TREES = 300                                    # equal tree budget for every variant, as in Step 8
BOOTSTRAPS = 2000
OUT = PROCESSED / 'feature_content_profile'
RUN = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + ('_smoke' if SMOKE else '')
RUN_DIR = OUT / 'runs' / RUN
RUN_DIR.mkdir(parents=True, exist_ok=False)
FEATURE_FILE = OUT / f"content_profile_features{'_smoke' if SMOKE else ''}.parquet"
print('data:', DATA_DIR, '| run:', RUN)

## 1. Candidates, history and the test query

Each user has exactly one slate (the test query), so `user` keys both the history and the slate. Candidates keep the `candidates.parquet` row order, which is also the row order of `features.parquet`. History events are kept in chronological order (`pos`) and include follows (`type == 3`), which count as a click-strength signal below.

In [ ]:
users = pd.read_parquet(PROCESSED / 'users.parquet', columns=['user', 'query'])
nU = len(users)
assert (users['user'].to_numpy() == np.arange(nU)).all()
in_scope = np.random.default_rng(SEED + 2).random(nU) < 0.02 if SMOKE else np.ones(nU, bool)

cands = pd.read_parquet(PROCESSED / 'candidates.parquet', columns=['user', 'wid', 'label'])
cands = cands[in_scope[cands['user'].to_numpy()]].reset_index(drop=True)
hist = pd.read_parquet(PROCESSED / 'events.parquet', columns=['user', 'pos', 'wid', 'type', 'age_days'])
hist = hist[in_scope[hist['user'].to_numpy()]].sort_values(['user', 'pos'], kind='stable').reset_index(drop=True)
assert (np.diff(cands['user'].to_numpy()) >= 0).all(), 'candidates.parquet is expected in user order'
assert hist['type'].isin([0, 1, 2, 3]).all() and hist['age_days'].ge(0).all()
print(f"users {in_scope.sum():,} | candidate rows {len(cands):,} | history events {len(hist):,}")

## 2. Item text: TF-IDF over name, brand and category terms

All text is anonymised term IDs separated by `\x18`, shared by queries and every product field. Each product becomes one document made of its name terms, its brand-name terms and its level-3 and level-4 category-name terms. Brand and category terms are separate vocabulary entries from name terms (key = `term · 3 + field`), so a shared brand or category adds similarity of its own even when the titles differ.

Weighting matches the exploration this feature was designed in (sklearn `TfidfVectorizer(sublinear_tf=True, min_df=2)`): tf = 1 + ln(count) (unlike Step 3's binary tf, a repeated title term counts a little more), idf = ln((N + 1) / (df + 1)) + 1, terms that occur in fewer than two documents are dropped, and rows are L2-normalised so a dot product is a cosine. It is built straight from the term-ID arrays as in Steps 3–4 so that millions of products fit in memory. Only products that appear in a slate or a history are parsed. TF-IDF uses no labels, so fitting it on candidate text is not leakage.

The test query is projected into the same space through its name-field terms; query terms that no product name contains are dropped.

In [ ]:
FIELDS = {'name': 0, 'brand_name': 1, 'cate_name_3': 2, 'cate_name_4': 2}   # levels 3 and 4 share the category field
KMAX = 1 << 34                                       # packs (doc in batch, key) into one int64
need = np.union1d(cands['wid'].to_numpy(), hist['wid'].to_numpy())

def tokens_of(col):
    # Arrow string column -> (tokens per row, flat int64 tokens); empty fields become -1.
    parts = pc.split_pattern(pc.fill_null(col, '-1'), '\x18')
    flat = pc.list_flatten(parts)
    flat = pc.if_else(pc.equal(flat, ''), '-1', flat)
    return pc.list_value_length(parts).to_numpy().astype(np.int64), pc.cast(flat, pa.int64()).to_numpy()

t0 = time.time()
wid_parts, doc_parts, key_parts, tf_parts = [], [], [], []
reader = pacsv.open_csv(DATA_DIR / 'product_meta_data.txt',
    parse_options=pacsv.ParseOptions(delimiter='\t', quote_char=False),
    convert_options=pacsv.ConvertOptions(include_columns=['wid'] + list(FIELDS),
        column_types={'wid': pa.int64(), **{c: pa.string() for c in FIELDS}}))
off = 0
for batch in reader:
    tb = pa.Table.from_batches([batch])
    wid = tb['wid'].to_numpy()
    j = np.searchsorted(need, wid)
    keep = (j < len(need)) & (need[np.minimum(j, len(need) - 1)] == wid)
    if not keep.any():
        continue
    tb = tb.filter(pa.array(keep)); n = len(tb)
    docs, keys = [], []
    for c, f in FIELDS.items():
        ln, tk = tokens_of(tb.column(c))
        docs.append(np.repeat(np.arange(n, dtype=np.int64), ln)); keys.append(tk * 3 + f)
    d, k = np.concatenate(docs), np.concatenate(keys)
    d, k = d[k >= 0], k[k >= 0]                      # the -1 placeholders give negative keys
    assert k.max(initial=0) < KMAX
    u, cnt = np.unique(d * KMAX + k, return_counts=True)   # term counts per document (a product is in one batch)
    wid_parts.append(tb['wid'].to_numpy()); doc_parts.append((u // KMAX + off).astype(np.int32))
    key_parts.append(u % KMAX); tf_parts.append(cnt.astype(np.float32))
    off += n

item_wid = np.concatenate(wid_parts)
assert len(np.unique(item_wid)) == len(item_wid), 'duplicate wid in product_meta_data.txt'
doc, key, tf = np.concatenate(doc_parts), np.concatenate(key_parts), np.concatenate(tf_parts)
del wid_parts, doc_parts, key_parts, tf_parts; gc.collect()
nD = len(item_wid)
vocab_all, col, df = np.unique(key, return_inverse=True, return_counts=True)
del key; gc.collect()
kept = df >= 2                                       # min_df = 2
VOCAB = vocab_all[kept]
IDF = (np.log((nD + 1) / (df[kept] + 1)) + 1).astype(np.float32)
new_col = (np.cumsum(kept) - 1).astype(np.int32)
m = kept[col]
doc, col, w = doc[m], new_col[col[m]], (1 + np.log(tf[m])) * IDF[new_col[col[m]]]
del tf, m, vocab_all, df, kept, new_col; gc.collect()
norm = np.sqrt(np.bincount(doc, weights=w.astype(np.float64) ** 2, minlength=nD)); norm[norm == 0] = 1
X = sp.csr_matrix((w / norm[doc].astype(np.float32), (doc, col)), shape=(nD, len(VOCAB)), dtype=np.float32)
del doc, col, w, norm; gc.collect()
item_index = pd.Index(item_wid)
print(f'TF-IDF built in {time.time() - t0:.0f}s: {nD:,} products x {len(VOCAB):,} terms, {X.nnz:,} non-zeros')
print(f"metadata coverage: candidate rows {(item_index.get_indexer(cands['wid']) >= 0).mean():.2%} | "
      f"history events {(item_index.get_indexer(hist['wid']) >= 0).mean():.2%}")

In [ ]:
# test-query vectors: name-field terms, same tf / idf / normalisation; one row per user
qs = users['query'].fillna('').str.split('\x18')
q_doc = np.repeat(users['user'].to_numpy().astype(np.int64), qs.str.len().to_numpy())
q_tok = pd.to_numeric(qs.explode(), errors='coerce').fillna(-1).to_numpy(np.int64)
ok = q_tok >= 0
u, cnt = np.unique(q_doc[ok] * KMAX + q_tok[ok] * 3 + FIELDS['name'], return_counts=True)
qd, qk = u // KMAX, u % KMAX
pos = np.searchsorted(VOCAB, qk)
inv = (pos < len(VOCAB)) & (VOCAB[np.minimum(pos, len(VOCAB) - 1)] == qk)
qd, pos, qw = qd[inv], pos[inv], (1 + np.log(cnt[inv])) * IDF[pos[inv]]
qn = np.sqrt(np.bincount(qd, weights=qw ** 2, minlength=nU)); qn[qn == 0] = 1
Q = sp.csr_matrix((qw / qn[qd], (qd, pos)), shape=(nU, len(VOCAB)), dtype=np.float32)
print(f'query term coverage in the name vocabulary: {inv.mean():.1%} | '
      f'users with an empty query vector: {(Q.getnnz(axis=1)[in_scope] == 0).mean():.2%}')
del qs, q_doc, q_tok, ok, u, cnt, qd, qk, pos, inv, qw, qn; gc.collect();

## 3. User × candidate similarity features

| Column | Definition |
|---|---|
| `cps_max` | cosine to the single most similar history product |
| `cps_mean` | cosine to one profile weighted by action strength (click 1, cart 2, order 3, follow 1) × 30-day half-life; the original single-column Feature 3, kept as the reference |
| `cps_query` | profile re-weighted towards the current query: weight = 30-day decay × (0.05 + cos(product, query))² |
| `cps_long`, `cps_short` | profile with a 90-day / 1-day half-life |
| `cps_top3_new` | mean of the 3 largest cosines to history products other than the candidate itself (negatives clipped to 0) |
| `cps_cart`, `cps_buy` | profile over cart + order events / order events only, 90-day half-life |
| `cps_max_cart`, `cps_max_ord` | nearest history product whose strongest action was a cart / an order |
| `cps_best_age` | ln(1 + days since the best-matching history product was last touched) |
| `slate_pct_cps_max`, `slate_pct_cps_query` | within-slate percentile ranks, like Step 4b's `slate_pct_*` |

**Computation.** For each user, S holds the cosines between the slate's candidates and the user's distinct history products, and G is the Gram matrix of those products. A profile p = Σ wᵢ xᵢ gives cos(x_c, p) = (S w)_c / √(wᵀ G w), so every profile column shares one sparse product per user. Candidates without metadata (and users with no history product that has metadata) get 0; `cps_best_age` gets NaN there, which LightGBM handles natively.

The decays use `age_days` from Step 1. Clicks only exist for ~7 days and carts for ~90, so the 1-day profile is click-led and the order-only profile reaches back the full year; the columns are kept separate rather than mixed into one fixed horizon for that reason.

In [ ]:
CPS_COLS = ['cps_max', 'cps_mean', 'cps_query', 'cps_long', 'cps_short', 'cps_top3_new', 'cps_cart', 'cps_buy',
            'cps_max_cart', 'cps_max_ord', 'cps_best_age']
PCT_SRC = ['cps_max', 'cps_query']
NEW = CPS_COLS + [f'slate_pct_{c}' for c in PCT_SRC]
LEVEL = np.array([1, 2, 3, 1])                      # action strength by type code: CLICK, CART, ORD, FLW
LN2 = np.log(2.0)

def decay(age_days, half_life_days):
    return np.exp(-LN2 * age_days / half_life_days)

def content_profile_features(c_user, c_wid, h_user, h_wid, h_type, h_age, X, item_index, Q, log_every=20_000):
    # -> {column: float32 array aligned with the candidate rows}; candidate and history arrays sorted by user.
    assert (np.diff(c_user) >= 0).all() and (np.diff(h_user) >= 0).all(), 'inputs must be sorted by user'
    users_ = np.unique(c_user)
    cb, ce = np.searchsorted(c_user, users_, 'left'), np.searchsorted(c_user, users_, 'right')
    hb, he = np.searchsorted(h_user, users_, 'left'), np.searchsorted(h_user, users_, 'right')
    ci, hi = item_index.get_indexer(c_wid), item_index.get_indexer(h_wid)        # -1 = no metadata
    item_wid_ = item_index.to_numpy()
    out = {c: np.zeros(len(c_user), np.float32) for c in CPS_COLS}
    out['cps_best_age'][:] = np.nan
    t0 = time.time()
    for k, u in enumerate(users_):
        if k and k % log_every == 0:
            print(f'  {k:,} / {len(users_):,} users ({time.time() - t0:.0f}s)')
        a, b = cb[k], ce[k]
        cidx = ci[a:b]; cv = cidx >= 0
        hidx = hi[hb[k]:he[k]]; ev = np.flatnonzero(hidx >= 0)
        if not cv.any() or len(ev) == 0:
            continue
        t, age = h_type[hb[k]:he[k]][ev], h_age[hb[k]:he[k]][ev]
        uh, inv = np.unique(hidx[ev], return_inverse=True)            # distinct history products
        Xh = X[uh]
        S = (X[cidx[cv]] @ Xh.T).toarray()                             # candidates x history products
        G = (Xh @ Xh.T).toarray()
        r = (Xh @ Q[u].T).toarray().ravel()[inv]                       # each event's relevance to the current query

        def profile(w_event):
            w = np.bincount(inv, weights=w_event, minlength=len(uh))
            norm = np.sqrt(max(w @ G @ w, 0.0))
            return S @ w / norm if norm > 0 else np.zeros(S.shape[0])

        level = np.zeros(len(uh)); np.maximum.at(level, inv, LEVEL[t])   # strongest action per product
        newest = np.full(len(uh), np.inf); np.minimum.at(newest, inv, age)
        S_new = np.where(c_wid[a:b][cv][:, None] == item_wid_[uh][None, :], -1.0, S)
        cols = {
            'cps_max': S.max(1),
            'cps_mean': profile(LEVEL[t] * decay(age, 30)),
            'cps_query': profile(decay(age, 30) * (0.05 + r) ** 2),
            'cps_long': profile(decay(age, 90)),
            'cps_short': profile(decay(age, 1)),
            'cps_top3_new': np.maximum(np.sort(S_new, 1)[:, -3:], 0).mean(1),
            'cps_cart': profile(np.isin(t, (1, 2)) * decay(age, 90)),
            'cps_buy': profile((t == 2) * decay(age, 90)),
            'cps_max_cart': np.where(level[None, :] == 2, S, 0).max(1),
            'cps_max_ord': np.where(level[None, :] == 3, S, 0).max(1),
            'cps_best_age': np.log1p(newest[S.argmax(1)]),
        }
        for c, v in cols.items():
            out[c][a:b][cv] = v
    return out

### 3.1 Correctness check on a hand-made example

Brute-force dense versions of the definitions above on three users: one with a repeated history product and a candidate it has already interacted with, one with no history, and one whose candidate has no metadata.

In [ ]:
rng = np.random.default_rng(0)
Xd = rng.random((4, 6)) * (rng.random((4, 6)) < 0.6); Xd[:, 0] += 0.1
Xd /= np.linalg.norm(Xd, axis=1, keepdims=True)
Qd = np.zeros((3, 6)); Qd[0, [1, 3]] = 1 / np.sqrt(2)
idx_t = pd.Index([10, 11, 12, 13])                  # wids that have "metadata"
c_u, c_w = np.array([0, 0, 0, 1, 2]), np.array([10, 13, 99, 11, 99])
h_u, h_w = np.array([0, 0, 0, 0, 2]), np.array([10, 11, 12, 11, 10])
h_t, h_a = np.array([0, 2, 1, 0, 2]), np.array([1.0, 10.0, 40.0, 2.0, 5.0])
got = content_profile_features(c_u, c_w, h_u, h_w, h_t, h_a, sp.csr_matrix(Xd), idx_t, sp.csr_matrix(Qd))

def brute_profile(xc, rows, wts):
    p = (wts[:, None] * Xd[rows]).sum(0)
    return xc @ p / np.linalg.norm(p) if np.linalg.norm(p) > 0 else 0.0

rows, wts_t, ages = np.array([0, 1, 2, 1]), h_t[:4], h_a[:4]
for i, cw in enumerate([10, 13]):                   # user 0's two candidates with metadata
    xc = Xd[cw - 10]
    sims = Xd[[0, 1, 2]] @ xc
    rel = Xd[rows] @ Qd[0]
    exp = {'cps_max': sims.max(),
           'cps_mean': brute_profile(xc, rows, LEVEL[wts_t] * decay(ages, 30)),
           'cps_query': brute_profile(xc, rows, decay(ages, 30) * (0.05 + rel) ** 2),
           'cps_long': brute_profile(xc, rows, decay(ages, 90)),
           'cps_short': brute_profile(xc, rows, decay(ages, 1)),
           'cps_top3_new': np.sort([s for w_, s in zip([10, 11, 12], sims) if w_ != cw] + [0] * (cw == 10))[-3:].mean(),
           'cps_cart': brute_profile(xc, rows, np.isin(wts_t, (1, 2)) * decay(ages, 90)),
           'cps_buy': brute_profile(xc, rows, (wts_t == 2) * decay(ages, 90)),
           'cps_max_cart': sims[2], 'cps_max_ord': sims[1],
           'cps_best_age': np.log1p([1.0, 2.0, 40.0][sims.argmax()])}
    for c, v in exp.items():
        assert np.isclose(got[c][i], v, atol=1e-5), (cw, c, got[c][i], v)
for c in CPS_COLS:                                  # no metadata / no history -> 0 (age NaN)
    for i in (2, 3, 4):
        assert (np.isnan(got[c][i]) if c == 'cps_best_age' else got[c][i] == 0), (i, c)
print('all hand-made checks passed')

## 4. Build the feature table

About 45 s per 20k users on the exploration sample, so expect several minutes for all users. The table is written to a temporary file, read back, then swapped in, so a crash cannot leave a truncated file behind.

In [ ]:
t0 = time.time()
f3 = content_profile_features(cands['user'].to_numpy(), cands['wid'].to_numpy(), hist['user'].to_numpy(),
                              hist['wid'].to_numpy(), hist['type'].to_numpy(), hist['age_days'].to_numpy(np.float64),
                              X, item_index, Q)
cpf = pd.DataFrame({'user': cands['user'].to_numpy(), 'wid': cands['wid'].to_numpy(), **f3})
for src in PCT_SRC:
    cpf[f'slate_pct_{src}'] = cpf.groupby('user')[src].rank(pct=True).to_numpy(np.float32)
del f3; gc.collect()
print(f'features built in {time.time() - t0:.0f}s')

FEATURE_FILE.parent.mkdir(parents=True, exist_ok=True)
tmp = FEATURE_FILE.with_suffix('.tmp.parquet')
cpf.to_parquet(tmp, index=False)
assert len(pd.read_parquet(tmp, columns=['user'])) == len(cpf)
os.replace(tmp, FEATURE_FILE)
print('saved', FEATURE_FILE, cpf.shape)
cpf[NEW].describe(percentiles=[0.5, 0.9]).T.round(3)

## 5. Signal check (validation slates, each feature alone)

Each column is used alone as a ranker on the validation slates, with the Step 8 metric code (graded NDCG@10, random tie-breaks in canonical order). AUC is computed within slates (only positive–negative pairs from the same slate are compared, as in the exploration), for any interaction (label > 0) and for purchases (label 3). The last column repeats the interaction AUC after removing candidates the user has already interacted with, since a product in the history is literally part of its own profile. Comparable existing columns are shown when `features.parquet` has them.

In [ ]:
def slate_metrics(d, score, k=10, seed=42):
    # Graded NDCG, any-hit and purchase-hit; NaN for undefined denominators (same as Step 8).
    assert k > 0 and len(d) > 0
    score = np.asarray(score, dtype=float)
    assert len(d) == len(score) and np.isfinite(score).all()
    canonical = np.lexsort((d.wid.to_numpy(), d.user.to_numpy()))
    tie = np.empty(len(d)); tie[canonical] = np.random.default_rng(seed).random(len(d))
    u, lab = d.user.to_numpy(), d.label.to_numpy(dtype=float)
    order = np.lexsort((tie, -score, u))
    us, ys = u[order], lab[order]
    start = np.r_[0, np.flatnonzero(us[1:] != us[:-1]) + 1]
    size = np.diff(np.r_[start, len(us)])
    gid = np.repeat(np.arange(len(start)), size)
    rank = np.arange(len(us)) - np.repeat(start, size)
    top = rank < k
    discount = np.where(top, 1 / np.log2(rank + 2), 0)
    dcg = np.bincount(gid, weights=(2 ** ys - 1) * discount)
    ideal = lab[np.lexsort((-lab, u))]
    idcg = np.bincount(gid, weights=(2 ** ideal - 1) * discount)
    pos = np.bincount(gid, weights=ys > 0) > 0
    buy = np.bincount(gid, weights=ys == 3) > 0
    hit = np.bincount(gid, weights=(ys > 0) & top) > 0
    hit_buy = np.bincount(gid, weights=(ys == 3) & top) > 0
    return pd.DataFrame({'ndcg10': np.divide(dcg, idcg, out=np.full(len(start), np.nan), where=idcg > 0),
                         'hr10': np.where(pos, hit.astype(float), np.nan),
                         'purchase_hr10': np.where(buy, hit_buy.astype(float), np.nan)},
                        index=pd.Index(us[start], name='user'))

def auc(user, y, s):
    # Within-slate ROC AUC: share of (positive, negative) pairs from the same slate ranked correctly; ties count half.
    f = pd.DataFrame({'u': np.asarray(user), 'y': np.asarray(y, bool), 's': np.asarray(s)})
    f['r'] = f.groupby('u')['s'].rank()
    g = f.groupby('u')['y'].agg(['sum', 'size'])
    rank_sum = f[f.y].groupby('u')['r'].sum().reindex(g.index, fill_value=0)
    pairs = (g['sum'] * (g['size'] - g['sum'])).sum()
    return (rank_sum - g['sum'] * (g['sum'] + 1) / 2).sum() / pairs if pairs else np.nan

split_of = pd.read_parquet(PROCESSED / 'features.parquet', columns=['user', 'split']).drop_duplicates('user')
split_of = split_of.set_index('user')['split']
import pyarrow.parquet as pq, pyarrow.dataset as ds
COMPARE = ['q_name_tfidf_sim', 'u_sem_lsa_7d', 'u_sem_lsa_all', 'slate_pct_sem', 'ucat3_days_since_any']
have = [c for c in COMPARE if c in pq.read_schema(PROCESSED / 'features.parquet').names]
va1 = cpf[cpf['user'].map(split_of).eq('val').to_numpy()].copy()
va1['label'] = cands['label'].to_numpy()[va1.index]
if have:
    import pyarrow.dataset as ds
    ex = ds.dataset(PROCESSED / 'features.parquet').to_table(columns=['user', 'wid'] + have,
                                                            filter=ds.field('split') == 'val').to_pandas()
    va1 = va1.merge(ex.drop_duplicates(['user', 'wid']), on=['user', 'wid'], how='left', validate='many_to_one')
seen = pd.MultiIndex.from_frame(hist[['user', 'wid']]).unique()
fresh = ~pd.MultiIndex.from_frame(va1[['user', 'wid']]).isin(seen)
lab, vu = va1['label'].to_numpy(), va1['user'].to_numpy()
rows = {}
for c in NEW + have:
    s = va1[c].to_numpy(np.float64)
    s = -s if c in ('cps_best_age', 'ucat3_days_since_any') else s       # more recent = better
    s = np.where(np.isnan(s), -1e9, s)
    m = slate_metrics(va1, s).mean()
    rows[c] = {'auc_any': auc(vu, lab > 0, s), 'auc_purchase': auc(vu, lab == 3, s), 'ndcg10': m['ndcg10'],
               'purchase_hr10': m['purchase_hr10'], 'auc_any_not_own_history': auc(vu[fresh], lab[fresh] > 0, s[fresh])}
single = pd.DataFrame(rows).T
single.to_csv(RUN_DIR / 'single_feature_signal.csv')
print(f'validation slates {va1.user.nunique():,} | rows {len(va1):,} | own-history candidates {1 - fresh.mean():.2%}')
single.round(4).sort_values('auc_any', ascending=False)

## 6. Matched LightGBM comparisons (train → validation)

Same protocol as Step 8: the existing user-level split, each slate shuffled once, identical rows for every variant, LambdaRank with `label_gain=[0, 1, 3, 7]`, a fixed 300-tree budget and three seeds. Changing the feature list is the only intervention. Seasonality columns stay excluded (Step 4c). Variants:

- `baseline`: every numeric column of `features.parquet`;
- `plus_cps`: baseline + all new columns;
- `plus_cps_minus_lsa`: as `plus_cps` but without the LSA profile columns (`u_sem_lsa_7d`, `u_sem_lsa_all`, `slate_pct_sem`), to test whether the TF-IDF profiles replace them;
- `plus_cps_mean_only`: baseline + `cps_mean`, i.e. the original single-column Feature 3.

Paired intervals resample whole validation slates on seed-averaged scores; they are exploratory and not adjusted for multiple comparisons. Slates are also split by whether their positives are all cold (product in no user's history), mixed or all warm, as in Steps 3–7.

In [ ]:
import pyarrow.dataset as ds
import lightgbm as lgb

EXCLUDED = {'user', 'wid', 'label', 'split', 'pos', 'position', 'candidate_position', 'candidate_index',
            'cat3_festival_lift', 'u_festival_share', 'ucat3_fest_ord', 'ucat3_fest_share'}
flt = ds.field('split').isin(['train', 'val'])
if SMOKE:
    flt = flt & ds.field('user').isin(pa.array(np.flatnonzero(in_scope).astype(np.int32)))
feat = ds.dataset(PROCESSED / 'features.parquet').to_table(filter=flt).to_pandas()
assert not feat.duplicated(['user', 'wid']).any(), 'Duplicate slate-item key'
assert not set(NEW) & set(feat.columns), 'a new column name already exists in features.parquet'
BASE = [c for c in feat if c not in EXCLUDED and pd.api.types.is_numeric_dtype(feat[c])]
n0 = len(feat)
feat = feat.merge(cpf, on=['user', 'wid'], how='left', validate='one_to_one')
assert len(feat) == n0 and feat['cps_max'].notna().all(), 'feature rows do not line up with features.parquet'
feat = feat.sort_values(['user', 'wid'], kind='stable')
feat = feat.iloc[np.lexsort((np.random.default_rng(SEED).random(len(feat)), feat.user.to_numpy()))].reset_index(drop=True)

LSA = [c for c in ['u_sem_lsa_7d', 'u_sem_lsa_all', 'slate_pct_sem'] if c in BASE]
VARIANTS = {'baseline': BASE, 'plus_cps': BASE + NEW}
if LSA:
    VARIANTS['plus_cps_minus_lsa'] = [c for c in BASE if c not in LSA] + NEW
VARIANTS['plus_cps_mean_only'] = BASE + ['cps_mean']
(RUN_DIR / 'feature_manifest.json').write_text(json.dumps({'base': BASE, 'new': NEW, 'variants': VARIANTS}, indent=2))
print(f'base features {len(BASE)} | new {len(NEW)} | variants {list(VARIANTS)}')

frames, excluded = {}, {}
for split in ['train', 'val']:
    d = feat[feat.split == split]
    keep = d.groupby('user').label.transform('max') > 0
    excluded[split] = {'slates_total': int(d.user.nunique()), 'slates_no_positive': int(d.loc[~keep, 'user'].nunique())}
    frames[split] = d.loc[keep].reset_index(drop=True)
tr, va = frames['train'], frames['val']
del feat, frames; gc.collect()
assert len(tr) and len(va) and set(tr.user).isdisjoint(va.user)
gtr = tr.groupby('user', sort=False).size().to_numpy()
print(excluded)

rows, val_scores, gain = [], {}, {}
for seed in SEEDS:
    for name, cols in VARIANTS.items():
        model = lgb.LGBMRanker(objective='lambdarank', label_gain=[0, 1, 3, 7],
            n_estimators=TREES, learning_rate=0.05, num_leaves=63, min_child_samples=200,
            subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
            random_state=seed, n_jobs=-1, verbose=-1)
        model.fit(tr[cols], tr.label, group=gtr)
        for split, d in [('train', tr), ('val', va)]:
            score = model.predict(d[cols])
            rows.append(dict(seed=seed, variant=name, split=split, features=len(cols), **slate_metrics(d, score).mean().to_dict()))
            if split == 'val':
                val_scores.setdefault(name, []).append(score)
        if name == 'plus_cps':
            gain[seed] = pd.Series(model.booster_.feature_importance('gain'), index=cols)
        print(seed, name, 'validation NDCG@10:', round(rows[-1]['ndcg10'], 4))
results = pd.DataFrame(rows)
results.to_csv(RUN_DIR / 'metrics_by_seed.csv', index=False)
results.groupby(['variant', 'split'])[['ndcg10', 'hr10', 'purchase_hr10']].agg(['mean', 'std']).round(4)

In [ ]:
def paired_interval(diff, draws=2000, seed=42):
    diff = np.asarray(diff, dtype=float)
    diff = diff[np.isfinite(diff)]
    if not len(diff):
        return dict(slates=0, delta=np.nan, lower95=np.nan, upper95=np.nan)
    rng = np.random.default_rng(seed)
    means = np.array([diff[rng.integers(len(diff), size=len(diff))].mean() for _ in range(draws)])
    lo, hi = np.quantile(means, [.025, .975])
    return dict(slates=len(diff), delta=float(diff.mean()), lower95=float(lo), upper95=float(hi))

averaged = {name: slate_metrics(va, np.mean(scores, axis=0)) for name, scores in val_scores.items()}
comparisons = []
for name, m in averaged.items():
    if name == 'baseline':
        continue
    assert m.index.equals(averaged['baseline'].index)
    for metric in ['ndcg10', 'hr10', 'purchase_hr10']:
        diff = (m[metric] - averaged['baseline'][metric]).dropna().to_numpy()
        comparisons.append(dict(variant=name, metric=metric, **paired_interval(diff, BOOTSTRAPS)))
comparison = pd.DataFrame(comparisons)
comparison.to_csv(RUN_DIR / 'paired_validation_comparisons.csv', index=False)
display(comparison.round(4))

share = pd.concat(gain, axis=1).mean(axis=1); share /= share.sum()
print('share of total split gain in plus_cps (mean over seeds):')
display(share.loc[NEW + LSA].sort_values(ascending=False).round(4).to_frame('gain_share'))

hist_wids = np.unique(pd.read_parquet(PROCESSED / 'events.parquet', columns=['wid'])['wid'].to_numpy())
j = np.searchsorted(hist_wids, va['wid'].to_numpy())
cold = (j >= len(hist_wids)) | (hist_wids[np.minimum(j, len(hist_wids) - 1)] != va['wid'].to_numpy())
cold_pos = pd.Series(np.where(va['label'].to_numpy() > 0, cold, np.nan)).groupby(va['user'].to_numpy()).mean()
seg_of = pd.cut(cold_pos, [-0.01, 0.0, 0.999, 1.0], labels=['all warm', 'mixed', 'all cold'])
segments = []
for name, m in averaged.items():
    for s in ['all warm', 'mixed', 'all cold']:
        part = m.loc[seg_of.reindex(m.index).eq(s).to_numpy()]
        segments.append(dict(variant=name, segment=s, slates=len(part), **part.mean().to_dict()))
segments = pd.DataFrame(segments)
segments.to_csv(RUN_DIR / 'cold_warm_segments.csv', index=False)
display(segments.pivot(index='variant', columns='segment', values='ndcg10').round(4))

manifest = {'run': RUN, 'smoke': SMOKE, 'seeds': SEEDS, 'trees': TREES, 'bootstrap_draws': BOOTSTRAPS,
    'excluded_slates': excluded, 'tfidf': {'products': int(nD), 'terms': int(len(VOCAB)), 'nnz': int(X.nnz)},
    'python': platform.python_version(),
    'packages': {p: importlib.metadata.version(p) for p in ['pandas', 'numpy', 'pyarrow', 'scipy', 'lightgbm']},
    'input_files': {p.name: {'bytes': p.stat().st_size, 'mtime_ns': p.stat().st_mtime_ns}
                    for p in [PROCESSED / 'features.parquet', PROCESSED / 'events.parquet',
                              PROCESSED / 'candidates.parquet', DATA_DIR / 'product_meta_data.txt']}}
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2))
print('saved results to', RUN_DIR)

In [ ]:
try:                                    # Colab: finish uploading the outputs before the runtime can close
    drive.flush_and_unmount()
    print('Drive flushed and unmounted - safe to close the runtime')
except NameError:
    pass